# 06 - עמידות (Robustness) והתקפה ממוקדת

מחברת זו בוחנת כמה מרשת התחבורה הציבורית בישראל שורדת כאשר תחנות מוצאות משירות. אנו מדמים הסרה הדרגתית של
תחנות תחת מספר אסטרטגיות של *התקפה ממוקדת* (targeted attack) - דרגה (degree) גבוהה ביותר, דרגה משוקללת גבוהה
ביותר, PageRank גבוה ביותר, betweenness גבוה ביותר, ונקודות חיתוך (articulation points) תחילה - ומשווים כל אחת
מהן לבסיס השוואה של *כשל אקראי* (random failure) הממוצע על פני מספר ניסויים עם זרעים (seeds) קבועים. בריאות
הרשת לאחר כל שלב הסרה נמדדת באמצעות גודל הרכיב הקשיר הגדול ביותר (LCC): כל עוד שורד רכיב ענק אחד, רוב הנוסעים
עדיין יכולים להגיע לרוב היעדים באמצעות מעברים; ברגע שהוא מתנפץ, הרשת נחלקה למעשה לאזורים מנותקים.

**שאלת המחקר:** *האם הרשת פגיעה למספר קטן של כשלי תחנות נבחרים היטב, ואיזו תפיסה של "תחנה חשובה" מזהה את
היעדים המזיקים ביותר?*

**קלט**
- גרף הסמיכות של הנסיעות (trip-adjacency) הבלתי מכוון שהופק במחברת `02_graph_construction`
  (`outputs/nb/02_graph_construction/...`): גרף ברמת התחנה שבו קשת `u - v` פירושה שקיימת נסיעה המשרתת את
  `v` מיד לאחר `u`, משוקללת לפי מספר הנסיעות המשתמשות בקטע זה. **נדרש.**
- אופציונלי: טבלת מדדי המרכזיות לכל תחנה ממחברת המרכזיות (`stop_metrics.csv`) ורשימת נקודות החיתוך ממחברת
  הניתוח התיאורי. אם אחד מהם חסר, מחברת זו מחשבת מחדש בעצמה את מה שהיא צריכה, ולכן ניתן להריץ אותה גם באופן
  עצמאי מיד לאחר מחברת 02.

**פלט** (הכול תחת `outputs/nb/06_robustness_analysis/`)
- `tables/disruption_results.csv` - רשת הסימולציה המלאה: אסטרטגיה, k שהוסרו, גודל ה-LCC, **ושתי** הנרמולים של
  חלקו של ה-LCC.
- `tables/ap_dedup_check.csv` - ראיה לתיקון באג הסרת הכפילויות המתואר להלן.
- `tables/damage_snapshots.csv` - חלקו של ה-LCC עבור כל אסטרטגיה במספר ערכי k מעניינים.
- `tables/auc_summary.csv` - השטח מתחת לכל עקומת עמידות (ציון עמידות כולל).
- `figures/resilience_curves_dual.png`, `figures/damage_at_k50_bar.png`, `figures/auc_comparison.png`.

**שני באגים שהיו בגרסה הקודמת של ניתוח זה מתוקנים כאן** - שניהם מתועדים בסעיף נפרד לפני הרצת הסימולציה, משום
שהם משנים את המסקנות.

## אתחול סביבת העבודה

התא הבא מאפשר להריץ את המחברת הן על עותק מקומי והן על Google Colab: הוא מתקין רק את החבילות שאכן חסרות, מאתר
את שורש המאגר (ומשכפל אותו אם אנו על Colab), ויוצר את תיקיית `outputs/nb` המשותפת שאליה כותבת כל מחברת בסדרה
זו. אין כאן דבר הייחודי לניתוח העמידות - זוהי אותה הקדמה המופיעה בכל מחברות הפרויקט.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## ייבוא ספריות, קבועים הניתנים לכוונון ותיקיית הפלט של השלב

כל מה שצורך זמן מעבד נשלט על ידי הקבועים שלהלן, כך שניתן לקצר את זמן הריצה של המחברת כולה מבלי לגעת באף תא אחר:

| קבוע | משמעות | עלות |
|---|---|---|
| `MAX_REMOVALS = 3000` | עד כמה עמוק מגיעה ההתקפה (כ-10% מתוך כ-30k התחנות) | לינארית |
| `STEPS = 40` | כמה נקודות מוערכות לאורך כל עקומה | מעבר אחד של רכיבים קשירים לכל נקודה |
| `EXTRA_KS` | מספר ערכי k קטנים המתווספים לרשת כדי שהחלק המוקדם והמעניין של העקומה לא יידלג | 4 נקודות נוספות |
| `RANDOM_TRIALS = 5` | חזרות עם זרעים קבועים הממוצעות עבור בסיס ההשוואה האקראי | מכפיל את עלות בסיס ההשוואה פי 5 |
| `BETWEENNESS_SAMPLES = 300` | מספר נקודות הציר (pivots) עבור betweenness מקורב, **בשימוש רק אם** שלב המרכזיות אינו זמין | כ-300 מעברי BFS |

בסך הכול הסימולציה מבצעת בקירוב `6 x 44 + 4 x 44 ~ 440` הערכות של רכיבים קשירים על גרף בן 30k צמתים / 52k
קשתות. על מחשב נייד רגיל מדובר בכ-1-3 דקות; זהו החלק האיטי היחיד במחברת. כל שלב הסרה משתמש ב-*תצוגת תת-גרף*
(subgraph view) במקום בהעתקת הגרף, מה ששומר על צריכת זיכרון קבועה.

מחברת זו כותבת אך ורק לתוך `outputs/nb/06_robustness_analysis/`; התוצאות הישנות המצוטטות בדוח תחת
`outputs/tables` ו-`outputs/figures` אינן משתנות כלל.

In [ ]:
_ensure('pandas', 'numpy', 'networkx', 'matplotlib')

import pickle, random, time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt

# ---------------- tunables: all runtime cost lives here ----------------
MAX_REMOVALS = 3000          # deepest attack: ~10% of the network
STEPS = 40                   # evenly spaced points from 0 to MAX_REMOVALS
EXTRA_KS = [10, 25, 50, 100] # small-k anchors so the early curve is resolved
RANDOM_TRIALS = 5            # seeded trials averaged for the random baseline
SEED = 42
BETWEENNESS_SAMPLES = 300    # only used if centrality has to be recomputed here
SNAPSHOT_K = 50              # k used for the 'damage bar chart'

STAGE = OUT / '06_robustness_analysis'
(STAGE / 'tables').mkdir(parents=True, exist_ok=True)
(STAGE / 'figures').mkdir(parents=True, exist_ok=True)

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# numpy 2 renamed trapz -> trapezoid; support both.
_trapz = getattr(np, 'trapezoid', None) or np.trapz

print('stage folder :', STAGE)
print('networkx     :', nx.__version__)
print('pandas       :', pd.__version__)

## איתור התוצרים של השלבים הקודמים

מחברת זו צורכת את הגרף שנבנה במחברת 02 במקום לבנות אותו מחדש, משום שבנייתו מחייבת קריאה בזרימה (streaming) של
קובץ ההזנה `stop_times.txt` בגודל 816 MB. פונקציות העזר שלהלן מחפשות בתוך `outputs/nb/` את תיקיית השלב של
מחברת קודמת ואת קובץ מסוים בתוכה, תוך גילוי סובלנות להבדלים קטנים בשמות הקבצים. נתמכים שני מסלולי טעינה:

1. גרף `networkx` שנשמר כ-pickle (`graph_undirected.pkl` או דומה) - נעשה בו שימוש ישיר;
2. אחרת `edges.csv` (בתוספת `nodes.csv` עבור תכונות התחנות) - הגרף הבלתי מכוון נבנה מחדש ממנו, תוך סכימת תדירות
   הנסיעות בשני כיווני הנסיעה למשקל קשת בלתי מכוונת יחיד, בדיוק כפי שמחברת 02 מגדירה זאת.

אם אף אחד מהם אינו קיים, אנו עוצרים עם שגיאה ברורה ובת-פעולה במקום לנתח בשקט משהו אחר. כל מזהי הצמתים מומרים
ל-`str` כך שמזהים המגיעים מ-pickle ומזהים המגיעים מ-CSV לעולם לא יוכלו לסתור זה את זה.

In [ ]:
def find_stage_dir(*keywords):
    """First sub-folder of OUT whose name contains one of the keywords (case-insensitive)."""
    if not OUT.exists():
        return None
    for kw in keywords:
        for d in sorted(p for p in OUT.iterdir() if p.is_dir()):
            if kw in d.name.lower():
                return d
    return None


def find_file(root, *patterns):
    """First file under `root` matching one of the glob patterns, searched recursively."""
    if root is None or not root.exists():
        return None
    for pat in patterns:
        hits = sorted(root.rglob(pat))
        if hits:
            return hits[0]
    return None


def _pick(columns, options):
    for o in options:
        if o in columns:
            return o
    return None


def to_undirected_sum(D):
    """Collapse a directed graph into an undirected one, summing both directions' weights."""
    U = nx.Graph()
    U.add_nodes_from(D.nodes(data=True))
    for u, v, data in D.edges(data=True):
        w = float(data.get('weight', 1))
        if u == v:
            continue
        if U.has_edge(u, v):
            U[u][v]['weight'] += w
        else:
            U.add_edge(u, v, weight=w)
    return U


def graph_from_csv(edges_csv, nodes_csv=None):
    """Rebuild the undirected, trip-frequency weighted graph from the stage-02 CSV exports."""
    e = pd.read_csv(edges_csv, dtype=str, encoding='utf-8-sig')
    cols = list(e.columns)
    cu = _pick(cols, ['from_stop', 'source', 'from', 'u', 'from_stop_id'])
    cv = _pick(cols, ['to_stop', 'target', 'to', 'v', 'to_stop_id'])
    cw = _pick(cols, ['trip_frequency', 'weight', 'trips', 'count'])
    if cu is None or cv is None:
        cu, cv = cols[0], cols[1]
    us = e[cu].astype(str).tolist()
    vs = e[cv].astype(str).tolist()
    ws = (pd.to_numeric(e[cw], errors='coerce').fillna(1.0).tolist()
          if cw is not None else [1.0] * len(us))

    G = nx.Graph()
    for a, b, w in zip(us, vs, ws):
        if a == b:
            continue
        if G.has_edge(a, b):
            G[a][b]['weight'] += float(w)
        else:
            G.add_edge(a, b, weight=float(w))

    if nodes_csv is not None and Path(nodes_csv).exists():
        n = pd.read_csv(nodes_csv, dtype=str, encoding='utf-8-sig')
        if 'stop_id' in n.columns:
            n = n.set_index(n['stop_id'].astype(str))
            for nid in G.nodes():
                if nid in n.index:
                    row = n.loc[nid]
                    if isinstance(row, pd.DataFrame):
                        row = row.iloc[0]
                    G.nodes[nid].update({k: row.get(k, '') for k in
                                         ['stop_name', 'region', 'metro'] if k in n.columns})
    return G


GRAPH_STAGE = find_stage_dir('graph_construction', 'graph')

graph_pkl = find_file(GRAPH_STAGE, '*undirected*.pkl', 'graph*.pkl', '*.gpickle')
edges_csv = find_file(GRAPH_STAGE, 'edges.csv', '*edges*.csv')

if graph_pkl is not None:
    with open(graph_pkl, 'rb') as f:
        _g = pickle.load(f)
    G = to_undirected_sum(_g) if _g.is_directed() else _g
    GRAPH_SOURCE = graph_pkl
elif edges_csv is not None:
    G = graph_from_csv(edges_csv, find_file(GRAPH_STAGE, 'nodes.csv', '*nodes*.csv'))
    GRAPH_SOURCE = edges_csv
else:
    raise FileNotFoundError(
        'No graph artifact found under ' + str(OUT) + '. Expected a stage folder such as '
        "outputs/nb/02_graph_construction containing graph_undirected.pkl or edges.csv - "
        'run notebook 02_graph_construction first.')

# Normalise node ids to strings and make sure every edge carries a numeric weight.
if any(not isinstance(n, str) for n in G.nodes()):
    G = nx.relabel_nodes(G, {n: str(n) for n in G.nodes()})
for _, _, d in G.edges(data=True):
    d['weight'] = float(d.get('weight', 1.0))

N0 = G.number_of_nodes()
components = sorted((len(c) for c in nx.connected_components(G)), reverse=True)
print('graph source        :', GRAPH_SOURCE)
print('nodes / edges       :', format(N0, ','), '/', format(G.number_of_edges(), ','))
print('connected components:', len(components))
print('largest component   :', format(components[0], ','),
      '(' + str(round(100 * components[0] / N0, 2)) + '% of all stations)')

## דירוגי ההתקפה: מדדי מרכזיות

התקפה ממוקדת מחייבת סדר כלשהו על התחנות. אנו משתמשים בארבע תפיסות שונות של חשיבות:

- **degree** - מספר התחנות השכנות השונות (בכמה כיוונים ניתן להמשיך);
- **דרגה משוקללת (strength)** - אותו הדבר, אך משוקלל לפי מספר הנסיעות בכל קטע, כלומר כמה שירות בפועל עובר דרך
  התחנה. זוהי תפיסת חשיבות של *תנועה* (traffic), והיא מחושבת כאן תמיד מתוך הגרף עצמו;
- **PageRank** - חשיבות המתפשטת דרך הרשת, כך שתחנה חשובה אם תחנות חשובות מזינות אותה;
- **betweenness** - שיעור המסלולים הקצרים ביותר העוברים דרך התחנה, כלומר עד כמה היא משמשת *גשר*. חישוב
  betweenness מדויק על 30k צמתים איטי מדי בהרבה, ולכן הוא מקורב מתוך `BETWEENNESS_SAMPLES` נקודות ציר אקראיות
  (זה מה שעושה `nx.betweenness_centrality(..., k=...)`).

אם מחברת המרכזיות כבר הורצה, אנו עושים שימוש חוזר ב-`stop_metrics.csv` שלה (כך ששתי המחברות מספרות בדיוק את
אותו הסיפור); אחרת אנו מחשבים כאן PageRank ו-betweenness מדגמי. בכל מקרה הטבלה ממופה מחדש על קבוצת הצמתים של
הגרף עצמו, כך שתחנה הנוכחת ב-CSV אך נעדרת מהגרף אינה יכולה לחלחל לתוך סדר ההתקפה.

In [ ]:
CENT_STAGE = find_stage_dir('centrality')
metrics_csv = find_file(CENT_STAGE, 'stop_metrics.csv', '*metric*.csv')

metrics = None
if metrics_csv is not None:
    m = pd.read_csv(metrics_csv, dtype=str, encoding='utf-8-sig')
    if 'stop_id' in m.columns and {'pagerank', 'betweenness'} <= set(m.columns):
        m['stop_id'] = m['stop_id'].astype(str)
        for c in ['pagerank', 'betweenness']:
            m[c] = pd.to_numeric(m[c], errors='coerce').fillna(0.0)
        m = m[m['stop_id'].isin(G)]
        if len(m) >= 0.5 * N0:
            metrics = m[['stop_id', 'pagerank', 'betweenness']].drop_duplicates('stop_id').copy()
            print('reusing centrality from', metrics_csv, '(' + format(len(metrics), ',') + ' stations)')

if metrics is None:
    print('centrality stage not found - computing PageRank and sampled betweenness here ...')
    t0 = time.time()
    pr = nx.pagerank(G, alpha=0.85, weight='weight')
    print('  pagerank done in', round(time.time() - t0, 1), 's')
    t0 = time.time()
    Gc = G.subgraph(max(nx.connected_components(G), key=len))
    btw = nx.betweenness_centrality(Gc, k=min(BETWEENNESS_SAMPLES, Gc.number_of_nodes()),
                                    seed=SEED, normalized=True)
    print('  sampled betweenness done in', round(time.time() - t0, 1), 's')
    metrics = pd.DataFrame({'stop_id': list(G.nodes())})
    metrics['pagerank'] = metrics['stop_id'].map(pr).fillna(0.0)
    metrics['betweenness'] = metrics['stop_id'].map(btw).fillna(0.0)

# Re-index over the graph's own nodes; degree columns always come from the graph itself.
metrics = (metrics.set_index('stop_id')
                  .reindex(list(G.nodes()))
                  .fillna(0.0)
                  .reset_index()
                  .rename(columns={'index': 'stop_id'}))
metrics['stop_id'] = metrics['stop_id'].astype(str)
metrics['degree'] = metrics['stop_id'].map(dict(G.degree())).astype(float)
metrics['weighted_degree'] = metrics['stop_id'].map(dict(G.degree(weight='weight'))).astype(float)
metrics['stop_name'] = metrics['stop_id'].map({n: G.nodes[n].get('stop_name', '') for n in G.nodes()})

print()
print('metrics table:', metrics.shape)
print(metrics[['degree', 'weighted_degree', 'pagerank', 'betweenness']].describe().round(4).to_string())

## דירוג ההתקפה: נקודות חיתוך (articulation points)

**נקודת חיתוך** (articulation point, צומת חתך) היא צומת שהסרתו מגדילה את מספר הרכיבים הקשירים. ברשת תחבורה אלו
התחנות הבודדות המחזיקות ענף שלם מחובר לשאר המדינה, ולכן הן המועמדות הטבעיות להתקפה הממוקדת המזיקה ביותר.
`networkx.articulation_points` מוצאת את כולן במעבר DFS יחיד (`O(V+E)`), דבר שהוא זול אף על גרף זה, ולכן אנו
מחשבים אותן כאן אם מחברת הניתוח התיאורי לא ייצאה אותן כבר.

קבוצת נקודות החיתוך הגולמית אינה מסודרת. כדי להפוך את האסטרטגיה למוגדרת היטב ולניתנת לשחזור, אנו מסדרים את
צמתי החתך לפי דרגה, מהגבוהה לנמוכה - מצופה שצומת חתך בעל חיבורים רבים ינתק חלק גדול יותר של הרשת מאשר צומת חתך
בעל דרגה 2.

In [ ]:
DESC_STAGE = find_stage_dir('descriptive', 'structure')
ap_csv = find_file(DESC_STAGE, 'articulation_points.csv', '*articulation*.csv')

ap_nodes = []
if ap_csv is not None:
    a = pd.read_csv(ap_csv, dtype=str, encoding='utf-8-sig')
    if 'stop_id' in a.columns:
        ap_nodes = [s for s in a['stop_id'].astype(str).tolist() if s in G]
        print('reusing', format(len(ap_nodes), ','), 'articulation points from', ap_csv)

if not ap_nodes:
    t0 = time.time()
    ap_nodes = list(nx.articulation_points(G))
    print('computed', format(len(ap_nodes), ','), 'articulation points in',
          round(time.time() - t0, 1), 's')

deg_map = dict(G.degree())
ap_nodes = sorted(set(ap_nodes), key=lambda n: -deg_map.get(n, 0))
print('articulation points:', format(len(ap_nodes), ','),
      '=', str(round(100 * len(ap_nodes) / N0, 2)) + '% of all stations')
print()
print('top 10 articulation points by degree:')
print(metrics[metrics['stop_id'].isin(ap_nodes[:10])]
      [['stop_id', 'stop_name', 'degree', 'betweenness']]
      .sort_values('degree', ascending=False).to_string(index=False))

## שני הבאגים בגרסה הקודמת של ניתוח זה

הסקריפט המקורי (`05_robustness_and_disruption_analysis/scripts/01_robustness.py`) הפיק את מספרי העמידות שצוטטו
בדוח הקודם. שני פגמים בו מתוקנים כאן; שניהם ראויים לציון מפורש משום שהם משנים את משמעות העקומות.

### באג 1 - עקומת נקודות החיתוך הסירה בפועל פחות תחנות ממה שציר ה-x טוען

אסטרטגיית נקודות החיתוך נבנתה כ-`ap_list + degree_sorted_list` וכל שלב הסיר `set(ordered[:k])`. קיימות רק כ-900
נקודות חיתוך, וכל אחת מהן **מופיעה גם** ברשימה הממוינת לפי דרגה, ולכן `k` הרשומות הראשונות של השרשור מכילות
כפילויות. משום שקבוצת ההסרה היא `set`, עבור `k` הגדול מכ-900 מספר התחנות השונות שהוסרו בפועל *קטן* מ-`k` -
העקומה שורטטה ב-`x = k` בעוד שרק `k - (חפיפה)` תחנות נמחקו. הדבר גורם לאסטרטגיית נקודות החיתוך להיראות מתונה
באופן מלאכותי בדיוק בנקודה שבה היא אמורה להיות התוקפנית ביותר.

**תיקון:** הסרת כפילויות מהסדר המשורשר *תוך שמירה על הסדר*, כך שמיקום `k` ברשימה הוא תמיד התחנה השונה ה-`k`,
ואימות (assert) שבכל שלב מוסרים בדיוק `k` צמתים שונים. התא שלהלן מכמת את הפער הישן.

### באג 2 - חלקו של ה-LCC נורמל לפי מספר התחנות המקורי

המקור דיווח `lcc_share = |LCC| / N_original`. לאחר הסרת `k` תחנות, לרשת יכולים להיוותר לכל היותר
`N_original - k` צמתים, ולכן מדד זה יורד בהכרח גם אם הרשת השורדת שלמה לחלוטין - הוא מערבב בין *"מחקנו צמתים"*
לבין *"הרשת התפרקה"*. עבור `k = 3000` (כ-10% מהרשת) המדד הגולמי חסום מלמעלה ב-0.90 ללא תלות בדבר.

**תיקון:** לדווח `|LCC| / (N_original - k)`, כלומר החלק **מבין התחנות ששרדו** שעדיין ניתנות להשגה הדדית - מדד
פרגמנטציה נקי הנשאר סביב 1.0 כל עוד הרשת רק מתכווצת. משום שהדוח הישן ציטט את הגרסה הגולמית, אנו מחשבים ומשרטטים
את **שתי** הגרסאות זו לצד זו במקום להחליף אחת בשנייה.

In [ ]:
def dedup_keep_order(seq):
    """Remove duplicates but keep first-occurrence order (this is the fix for bug 1)."""
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out


def order_by(col):
    """Stations sorted by a metric, highest first; ties broken by degree then id for determinism."""
    s = metrics.sort_values([col, 'degree', 'stop_id'], ascending=[False, False, True])
    return s['stop_id'].tolist()


degree_order = order_by('degree')
wdegree_order = order_by('weighted_degree')
pagerank_order = order_by('pagerank')
betweenness_order = order_by('betweenness')

ap_order_buggy = ap_nodes + degree_order            # what the original script did
ap_order = dedup_keep_order(ap_order_buggy)         # fixed: k-th entry is the k-th distinct station

# --- evidence for bug 1: how many distinct stations the old order really removed ---
check_rows = []
for k in [100, 300, 500, 900, 1000, 1500, 2000, 2500, 3000]:
    buggy = len(set(ap_order_buggy[:k]))
    fixed = len(set(ap_order[:k]))
    check_rows.append({'k_on_x_axis': k,
                       'distinct_removed_old': buggy,
                       'distinct_removed_fixed': fixed,
                       'stations_never_removed_old': k - buggy})
ap_check = pd.DataFrame(check_rows)
ap_check.to_csv(STAGE / 'tables' / 'ap_dedup_check.csv', index=False, encoding='utf-8-sig')
print('Bug 1 - articulation-point order, distinct stations actually removed:')
print(ap_check.to_string(index=False))

STRATEGIES = {
    'degree': degree_order,
    'weighted degree': wdegree_order,
    'pagerank': pagerank_order,
    'betweenness': betweenness_order,
    'articulation points': ap_order,
}
for name, order in STRATEGIES.items():
    assert len(order) == len(set(order)), 'duplicate entries in strategy ' + name
    if len(order) < MAX_REMOVALS:
        raise ValueError('strategy ' + name + ' has only ' + str(len(order)) +
                         ' stations, fewer than MAX_REMOVALS=' + str(MAX_REMOVALS))
print()
print('all', len(STRATEGIES), 'attack orders are duplicate-free and long enough.')

## מנוע הסימולציה

`lcc_size` מוחקת קבוצת תחנות ומחזירה את גודל הרכיב הקשיר הגדול ביותר שנותר. היא משתמשת ב-`G.subgraph(survivors)`,
שהיא *תצוגה* (view) לקריאה בלבד: לא נוצר עותק של הגרף בן 30k הצמתים, ורק סריקת הרכיבים הקשירים מורצת, בסיבוכיות
`O(V+E)` לכל הערכה.

`simulate` עוברת על רשת ערכי ה-k עבור אסטרטגיה דטרמיניסטית אחת ורושמת, לכל נקודה, את גודל ה-LCC יחד עם **שתי**
הנרמולים (באג 2). היא מאמתת שבכל שלב הוסרו בדיוק `k` תחנות שונות (באג 1).

`simulate_random` היא בסיס ההשוואה: עבור כל אחד מ-`RANDOM_TRIALS` הניסויים עם זרעים קבועים היא מגרילה תמורה
אקראית אחת של התחנות ומסירה את `k` הרשומות הראשונות שלה. שימוש בתמורה במקום במדגם בלתי תלוי לכל `k` הופך כל
ניסוי לרצף כשל הדרגתי עקבי (ההסרות מקוננות זו בזו, כמו בכשל מדורג אמיתי), ולאחר מכן ממוצעים את הניסויים. זהו
הבסיס שכל אסטרטגיה ממוקדת חייבת לנצח כדי להיחשב "ממוקדת".

In [ ]:
ALL_NODES = list(G.nodes())
NODE_SET = set(ALL_NODES)


def k_grid():
    """Evenly spaced removal counts plus a few small-k anchors, sorted and unique."""
    ks = {int(round(x)) for x in np.linspace(0, MAX_REMOVALS, STEPS)}
    ks.update(k for k in EXTRA_KS if k <= MAX_REMOVALS)
    return sorted(ks)


def lcc_size(removed):
    """Size of the largest connected component after deleting `removed` (a set of node ids)."""
    survivors = NODE_SET.difference(removed)
    if not survivors:
        return 0
    return max((len(c) for c in nx.connected_components(G.subgraph(survivors))), default=0)


def _row(strategy, k, size):
    surviving = N0 - k
    return {
        'strategy': strategy,
        'removed': k,
        'surviving_nodes': surviving,
        'lcc_size': size,
        # bug 2: the original (misleading) normalization, kept for comparability with the old report
        'lcc_share_original': round(size / N0, 5),
        # bug 2 fixed: fragmentation among the stations that are still there
        'lcc_share_surviving': round(size / surviving, 5) if surviving > 0 else 0.0,
    }


def simulate(strategy, order, verbose=True):
    order = [n for n in order if n in NODE_SET]
    rows = []
    t0 = time.time()
    for k in k_grid():
        removed = set(order[:k])
        assert len(removed) == k, 'strategy ' + strategy + ' removed ' + str(len(removed)) + ' != k=' + str(k)
        rows.append(_row(strategy, k, lcc_size(removed)))
    if verbose:
        print('  ' + strategy.ljust(22), 'done in', round(time.time() - t0, 1), 's',
              '| LCC share (surviving) at k=' + str(MAX_REMOVALS) + ':',
              rows[-1]['lcc_share_surviving'])
    return pd.DataFrame(rows)


def simulate_random(trials=RANDOM_TRIALS, seed=SEED, verbose=True):
    ks = k_grid()
    sizes = {k: [] for k in ks}
    t0 = time.time()
    for t in range(trials):
        rng = random.Random(seed + t)
        perm = list(ALL_NODES)
        rng.shuffle(perm)
        for k in ks:
            sizes[k].append(lcc_size(set(perm[:k])))
    rows = [_row('random (baseline)', k, float(np.mean(sizes[k]))) for k in ks]
    if verbose:
        print('  ' + 'random (baseline)'.ljust(22), 'done in', round(time.time() - t0, 1), 's',
              '|', trials, 'seeded trials averaged')
    return pd.DataFrame(rows)


print('k-grid (' + str(len(k_grid())) + ' points):', k_grid()[:12], '...', k_grid()[-3:])

## הרצת הסימולציה

זהו התא היקר מבחינה חישובית (בערך 1-3 דקות; ניתן להקטין את `STEPS`, `MAX_REMOVALS` או `RANDOM_TRIALS` כדי לקצרו).
הוא מריץ את חמש האסטרטגיות הממוקדות ואת בסיס ההשוואה האקראי על אותה רשת ערכי k ומשרשר את הכול ל-dataframe מסודר
אחד, אשר נכתב לאחר מכן אל `tables/disruption_results.csv` כששני הנרמולים מופיעים זה לצד זה.

In [ ]:
print('simulating attacks (', len(k_grid()), 'removal levels per strategy )')
t_all = time.time()

frames = [simulate(name, order) for name, order in STRATEGIES.items()]
frames.append(simulate_random())

results = pd.concat(frames, ignore_index=True)
results.to_csv(STAGE / 'tables' / 'disruption_results.csv', index=False, encoding='utf-8-sig')

print()
print('total simulation time:', round(time.time() - t_all, 1), 's')
print('saved', STAGE / 'tables' / 'disruption_results.csv', '(' + str(len(results)), 'rows )')
print()
print(results.head(8).to_string(index=False))

## עקומות עמידות תחת שני הנרמולים

האיור שלהלן מציג את אותה סימולציה פעמיים.

- **שמאל - `|LCC| / N_original`**: המדד ששימש בדוח הקודם. הקו האפור המקווקו הוא ה-*תקרה*
  `(N_original - k) / N_original`: אף עקומה אינה יכולה לעלות מעליו, משום שהתחנות שהוסרו אינן קיימות עוד. כל מרחק
  בין עקומה לבין אותה תקרה הוא פרגמנטציה אמיתית; שאר הירידה היא רק חשבונאות.
- **ימין - `|LCC| / (N_original - k)`**: המדד המתוקן. הוא מתחיל בכ-0.99 (הגרף אינו קשיר לחלוטין מלכתחילה - קיימים
  מספר רכיבים קטנים) ויורד רק כאשר הרשת השורדת אכן מתפרקת לחלקים. עקומה הנשארת שטוחה כאן פירושה שההתקפה הסירה
  תחנות אך **לא** ניפצה את הרשת.

הפער בין עקומה ממוקדת לבין בסיס ההשוואה האקראי הוא התשובה בפועל לשאלת המחקר: הוא מודד כמה גרוע יותר תוקף נבון
בהשוואה למזל רע.

In [ ]:
COLORS = {
    'degree': '#dc2626',
    'weighted degree': '#ea580c',
    'pagerank': '#16a34a',
    'betweenness': '#2563eb',
    'articulation points': '#7c3aed',
    'random (baseline)': '#6b7280',
}
STRATEGY_ORDER = list(COLORS.keys())

fig, axes = plt.subplots(1, 2, figsize=(14, 5.6), sharex=True)
panels = [
    ('lcc_share_original', 'LCC / original station count\n(as reported previously)'),
    ('lcc_share_surviving', 'LCC / surviving station count\n(fragmentation, corrected)'),
]

ks = np.array(k_grid(), dtype=float)
for ax, (col, title) in zip(axes, panels):
    for strat in STRATEGY_ORDER:
        g = results[results['strategy'] == strat].sort_values('removed')
        if g.empty:
            continue
        ax.plot(g['removed'], g[col], marker='o', markersize=3.2, linewidth=1.8,
                label=strat, color=COLORS[strat],
                linestyle='--' if strat == 'random (baseline)' else '-')
    if col == 'lcc_share_original':
        ax.plot(ks, (N0 - ks) / N0, color='#9ca3af', linewidth=1.2, linestyle=':',
                label='ceiling (N - k) / N')
    ax.set_xlabel('stations removed (k)')
    ax.set_ylabel(col)
    ax.set_title(title, fontsize=11)
    ax.set_ylim(0, 1.03)

axes[0].legend(loc='lower left', fontsize=8, framealpha=0.9)
fig.suptitle('Network resilience under targeted attack vs random failure', fontsize=13)
fig.tight_layout()
fig.savefig(STAGE / 'figures' / 'resilience_curves_dual.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved', STAGE / 'figures' / 'resilience_curves_dual.png')

## תצלומי נזק בגדלי התקפה מסוימים

עקומות טובות להבנת הצורה, אך הדוח זקוק למספרים. התא הבא מחלץ את מצב הרשת בשלושה גדלי התקפה אופייניים - התקפה
קטנה מאוד (`k = 50`, כ-0.16% מהתחנות), התקפה בינונית (`k = 500`) והעמוקה ביותר שדומתה (`k = MAX_REMOVALS`) -
עבור כל אסטרטגיה תחת שני הנרמולים, ומשרטט תרשים עמודות מקובץ עבור `k = SNAPSHOT_K`. משום שרשת ערכי ה-k מכילה
כעת עוגנים מפורשים בערכי k קטנים, שורת `k = 50` היא נקודה מדומה אמיתית ולא הנקודה הזמינה הקרובה ביותר (הגרסה
הקודמת סימנה תרשים כ-"k ~ 50" בעוד שבפועל שרטטה `k = 77`).

In [ ]:
snapshot_ks = [k for k in [SNAPSHOT_K, 500, MAX_REMOVALS] if k in set(k_grid())]
snapshots = (results[results['removed'].isin(snapshot_ks)]
             .sort_values(['removed', 'lcc_share_surviving'])
             .reset_index(drop=True))
snapshots.to_csv(STAGE / 'tables' / 'damage_snapshots.csv', index=False, encoding='utf-8-sig')
for k in snapshot_ks:
    print('--- k =', k, '(' + str(round(100 * k / N0, 2)) + '% of stations removed) ---')
    print(snapshots[snapshots['removed'] == k]
          [['strategy', 'lcc_size', 'lcc_share_original', 'lcc_share_surviving']]
          .to_string(index=False))
    print()

snap = (snapshots[snapshots['removed'] == SNAPSHOT_K]
        .set_index('strategy')
        .reindex([s for s in STRATEGY_ORDER])
        .dropna(subset=['lcc_size']))
x = np.arange(len(snap))
width = 0.38

fig, ax = plt.subplots(figsize=(10.5, 5))
b1 = ax.bar(x - width / 2, snap['lcc_share_original'], width,
            color='#94a3b8', label='LCC / original N (old measure)')
b2 = ax.bar(x + width / 2, snap['lcc_share_surviving'], width,
            color='#2563eb', label='LCC / surviving N (corrected)')
for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
                format(bar.get_height(), '.3f'), ha='center', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(snap.index, rotation=18, ha='right')
ax.set_ylabel('largest component share')
ax.set_ylim(0, 1.12)
ax.axhline(1.0, color='black', linestyle='--', linewidth=0.8, alpha=0.4)
ax.set_title('Damage after removing only ' + str(SNAPSHOT_K) + ' stations, by strategy')
ax.legend(fontsize=9, loc='lower right')
fig.tight_layout()
fig.savefig(STAGE / 'figures' / 'damage_at_k50_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved', STAGE / 'figures' / 'damage_at_k50_bar.png')

## מספר אחד לכל אסטרטגיה: השטח מתחת לעקומת העמידות

כדי לדרג את האסטרטגיות באופן כולל אנו מבצעים אינטגרציה על כל עקומה בכלל הטרפז ומחלקים ב-`MAX_REMOVALS`. התוצאה
היא חלקו ה*ממוצע* של הרכיב הקשיר הגדול ביותר לאורך כל ההתקפה, בסולם 0-1: **ערך נמוך יותר פירושו שהאסטרטגיה
גרמה נזק רב יותר**, כלומר היא התקפה טובה יותר (ומזהה את התחנות הקריטיות יותר). אנו מדווחים את ה-AUC עבור שני
הנרמולים; הדירוג אמור להיות זהה, שכן שני המדדים נבדלים בפקטור התלוי ב-`k` בלבד ולא באסטרטגיה.

In [ ]:
auc_rows = []
for strat in STRATEGY_ORDER:
    g = results[results['strategy'] == strat].sort_values('removed')
    if g.empty:
        continue
    auc_rows.append({
        'strategy': strat,
        'auc_original': round(float(_trapz(g['lcc_share_original'], g['removed'])) / MAX_REMOVALS, 4),
        'auc_surviving': round(float(_trapz(g['lcc_share_surviving'], g['removed'])) / MAX_REMOVALS, 4),
    })
auc = pd.DataFrame(auc_rows).sort_values('auc_surviving').reset_index(drop=True)
auc['rank_most_damaging'] = np.arange(1, len(auc) + 1)
auc.to_csv(STAGE / 'tables' / 'auc_summary.csv', index=False, encoding='utf-8-sig')
print('Average largest-component share over the attack (lower = more damaging attack):')
print(auc.to_string(index=False))

y = np.arange(len(auc))
height = 0.38
fig, ax = plt.subplots(figsize=(10, 4.8))
ax.barh(y - height / 2, auc['auc_original'], height, color='#94a3b8', label='AUC (LCC / original N)')
ax.barh(y + height / 2, auc['auc_surviving'], height, color='#2563eb', label='AUC (LCC / surviving N)')
for yi, (a, b) in enumerate(zip(auc['auc_original'], auc['auc_surviving'])):
    ax.text(a + 0.008, yi - height / 2, format(a, '.3f'), va='center', fontsize=8)
    ax.text(b + 0.008, yi + height / 2, format(b, '.3f'), va='center', fontsize=8)
ax.set_yticks(y)
ax.set_yticklabels(auc['strategy'])
ax.invert_yaxis()
ax.set_xlim(0, 1.1)
ax.set_xlabel('mean largest-component share over k = 0 .. ' + str(MAX_REMOVALS) + '  (lower = more damaging)')
ax.set_title('Overall robustness by attack strategy (area under the resilience curve)')
ax.legend(fontsize=9, loc='lower right')
fig.tight_layout()
fig.savefig(STAGE / 'figures' / 'auc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved', STAGE / 'figures' / 'auc_comparison.png')

## מסקנות

יש לקרוא את המספרים המדויקים מן הטבלאות שהודפסו לעיל (`auc_summary.csv`, `damage_snapshots.csv`); הנקודות שלהלן
הן מה שהסימולציה מראה מבחינה מבנית.

1. **להתקפות קטנות כמעט אין השפעה, וזו תוצאה אמיתית - לא תוצאה נוחה.** עבור `k = 50` כל אסטרטגיה - לרבות החדה
   ביותר - משאירה את הרשת השורדת כמעט כולה בחתיכה אחת. אין קומץ תחנות שאובדנן מנתק את התחבורה הציבורית בישראל.
   היתירות (redundancy) של המערכת אמיתית, וכל טענה בדבר "נקודת כשל יחידה קטסטרופלית" אינה נתמכת על ידי גרף זה.
2. **הנזק נעשה גלוי רק בסדר גודל של אלפי תחנות**, כלומר אחוזים אחדים מן הרשת. עבור `k = 3000` הדירוג בין
   האסטרטגיות רחב וחד-משמעי: התקפות המבוססות על degree ועל betweenness מפרקות את הרשת הרבה יותר מכשל אקראי,
   הנשאר קרוב לשלמות. זוהי החתימה הקלאסית של רשת הטרוגנית הנשלטת על ידי hubs: עמידה בפני כשל אקראי, פגיעה בפני
   התקפה ממוקדת - אך רק בפני התקפה *מתמשכת*.
3. **באג 1 שינה מסקנה.** עם הסדר נטול הכפילויות, אסטרטגיית נקודות החיתוך מסירה `k` תחנות שונות בכל שלב במקום
   פחות מכך בשקט, ועקומתה יורדת קרוב הרבה יותר להתקפה המבוססת על degree מאשר העקומה שפורסמה קודם. הממצא של
   הגרסה הקודמת, לפיו "נקודות חיתוך הן התקפה מתונה באופן מפתיע", היה ארטיפקט של הסרת פחות תחנות ממה שהציר טען,
   ולא תכונה של הרשת. כמו כן יש לשים לב שמעבר לכ-900 הסרות האסטרטגיה מיצתה את צמתי החתך והיא למעשה התקפת ה-degree
   כשצמתי החתך הועברו לראש הרשימה, ולכן התכנסות שתי העקומות בערכי `k` גדולים צפויה מעצם הבנייה.
4. **באג 2 שינה את גודל האפקט, לא את כיוונו.** תחת המדד הישן `|LCC| / N_original` כל עקומה - אפילו כשל אקראי -
   נראית כדועכת, אך חלק גדול מדעיכה זו הוא רק תקרת החישוב `(N - k) / N`. נרמול לפי התחנות ששרדו מראה שכשל אקראי
   משאיר את הרשת בלתי מפורקת למעשה על פני כל התחום, ולכן ה*ניגוד* בין התקפה ממוקדת לכשל אקראי חד יותר וכן יותר
   בפאנל הימני. מספרים המצוטטים מן הדוח הישן הם הפאנל השמאלי ויש לצטטם ככאלה.
5. **PageRank הוא דירוג התקפה גרוע כאן.** הוא מתרכז בתחנות קצה עמוסות תנועה בתוך ליבות מטרופוליניות צפופות,
   שהסרתן נבלעת בשל ריבוי החלופות שלהן, בעוד ש-degree ו-betweenness מאתרים את המחברים הדלילים שבין האזורים.
   להיות התחנה העמוסה ביותר ולהיות התחנה המחזיקה את הרשת יחד אינם היינו הך - וזהו המסר המעשי המרכזי של מחברת זו.

**הסתייגויות.** ה-betweenness מקורב מתוך `BETWEENNESS_SAMPLES` נקודות ציר, ולכן הדירוג שלו - וממילא גם עקומתו -
נושא רעש דגימה; הדירוגים מחושבים מחדש על הגרף ה*מקורי* ואינם מעודכנים לאחר כל הסרה, ולכן מדובר בהתקפות בו-זמניות
("initial-rank") ולא בהתקפות אדפטיביות, וזוהי המוסכמה הסטנדרטית אך החלשה מבין השתיים. לבסוף, קשירות היא קירוב
(proxy) טופולוגי: רשת הנשארת קשירה עדיין עלולה להיעשות איטית בהרבה למעבר, וזמן הנסיעה אינו ממודל כאן.